# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAliEjaz1/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Looking at position, impressions, and CTR before deciding anything — CTR in
particular has a heavy tail toward zero (confirmed in Week 3: 83%+ of a
sample had ctr=0), so any signal test needs to account for that.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{hf_token}');")
BASE = "hf://datasets/FlyRank/internship-warehouse"
path = f"{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet"

dist = con.execute(f"""
    SELECT gsc_avg_position, gsc_impressions, gsc_clicks
    FROM read_parquet('{path}')
    WHERE gsc_data_available IS TRUE
""").df()
dist["ctr"] = np.where(dist["gsc_impressions"] > 0, dist["gsc_clicks"] / dist["gsc_impressions"], 0.0)
print(dist[["gsc_avg_position", "gsc_impressions", "ctr"]].describe())
print(f"\nZero-CTR share: {(dist['ctr'] == 0).mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       gsc_avg_position  gsc_impressions           ctr
count      3.611061e+06     3.611061e+06  3.611061e+06
mean       1.582665e+01     7.772164e+01  3.080748e-03
std        1.985603e+01     2.498747e+02  3.009151e-02
min        0.000000e+00     1.000000e+00  0.000000e+00
25%        3.742120e+00     4.000000e+00  0.000000e+00
50%        7.500000e+00     1.600000e+01  0.000000e+00
75%        2.020000e+01     6.200000e+01  0.000000e+00
max        4.980000e+02     4.008400e+04  1.000000e+00

Zero-CTR share: 88.4%


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Signal 1 — CTR vs. position:** does CTR fall as position worsens?
**Signal 2 — quick-win zone volume:** does position 11-20 hold a meaningful,
impression-rich pool?
**Signal 3 — impressions vs. clicks:** do high-impression pages reliably
convert to more clicks, or does that break down at scale (new this
notebook)?

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sig1 = con.execute(f"""
    SELECT CASE WHEN gsc_avg_position<=3 THEN '1-3' WHEN gsc_avg_position<=10 THEN '4-10'
                WHEN gsc_avg_position<=20 THEN '11-20' ELSE '21+' END AS tier,
           COUNT(*) AS n, AVG(gsc_clicks*1.0/gsc_impressions) AS mean_ctr
    FROM read_parquet('{path}') WHERE gsc_data_available IS TRUE AND gsc_impressions>0
    GROUP BY tier ORDER BY tier
""").df()
print("Signal 1 (CTR vs position):"); print(sig1)
print("Verdict: CONFIRMED — CTR declines monotonically as position worsens.\n")

sig2 = con.execute(f"""
    SELECT CASE WHEN gsc_avg_position BETWEEN 11 AND 20 THEN 'quick_win' ELSE 'other' END AS zone,
           COUNT(*) AS n, AVG(gsc_impressions) AS mean_imp
    FROM read_parquet('{path}') WHERE gsc_data_available IS TRUE GROUP BY zone
""").df()
print("Signal 2 (quick-win volume):"); print(sig2)
print("Verdict: MIXED — real pool exists but not unusually rich vs other tiers.\n")

sig3 = dist[["gsc_impressions", "gsc_clicks"]].corr().iloc[0, 1]
print(f"Signal 3 (impressions vs clicks correlation): {sig3:.3f}")
print("Verdict: CONFIRMED — strong positive correlation, but see Section 3 for the catch.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 (CTR vs position):
    tier        n  mean_ctr
0    1-3   727362  0.004756
1  11-20   519223  0.002770
2    21+   908354  0.001289
3   4-10  1456122  0.003473
Verdict: CONFIRMED — CTR declines monotonically as position worsens.

Signal 2 (quick-win volume):
        zone        n   mean_imp
0      other  3158813  80.850698
1  quick_win   452248  55.866146
Verdict: MIXED — real pool exists but not unusually rich vs other tiers.

Signal 3 (impressions vs clicks correlation): 0.586
Verdict: CONFIRMED — strong positive correlation, but see Section 3 for the catch.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

FlyRank's refresh flags lean on staleness (days since last update). Testing
whether the impressions-clicks relationship actually holds at the low end,
where most declining pages sit — since a strong overall correlation can hide
a weak relationship in the segment that matters most for triage.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
low_imp = dist[dist["gsc_impressions"] < dist["gsc_impressions"].median()]
low_corr = low_imp[["gsc_impressions", "gsc_clicks"]].corr().iloc[0, 1]
print(f"Impressions-clicks correlation, below-median-impressions pages only: {low_corr:.3f}")
print(f"(n={len(low_imp)})")
print("Verdict: OPPOSITE-OF-EXPECTED STRENGTH — the strong portfolio-wide correlation")
print("(0.586) collapses to near-zero (0.071) among below-median-impression pages.")
print("The impressions-clicks relationship that looks strong overall is barely present")
print("in the low-traffic segment a real review queue actually draws from.")

Impressions-clicks correlation, below-median-impressions pages only: 0.071
(n=1799187)
Verdict: OPPOSITE-OF-EXPECTED STRENGTH — the strong portfolio-wide correlation
(0.586) collapses to near-zero (0.071) among below-median-impression pages.
The impressions-clicks relationship that looks strong overall is barely present
in the low-traffic segment a real review queue actually draws from.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team should trust the CTR-vs-position relationship (Signal 1) as a
reliable, monotonic pattern. The "quick-win zone" volume claim (Signal 2)
needs softer framing. Most importantly: the impressions-clicks correlation
that looks strong across the whole portfolio (0.586) nearly disappears
(0.071) among below-median-impression pages — exactly the pages a real
review queue mostly contains. A signal that looks strong in aggregate can be
almost useless in the specific segment where it will actually be applied;
always re-check a signal on that segment, not just the full dataset.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
takeaways = [
    "CTR vs position: reliable, monotonic — trust it",
    "quick-win zone volume: real but not exceptional — soften the framing",
    "portfolio-wide correlations can overstate strength in the low-traffic segment that matters most",
]
for t in takeaways:
    print(f"- {t}")

- CTR vs position: reliable, monotonic — trust it
- quick-win zone volume: real but not exceptional — soften the framing
- portfolio-wide correlations can overstate strength in the low-traffic segment that matters most


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.